# 🎨 Image Processing & Color Detection — Step-by-Step Tutorial

This notebook teaches **every technique used in `color_detector_app.py`**, one step at a time:

1. Loading and representing images as NumPy arrays
2. Understanding RGB pixels and HEX codes
3. Building a **60-color palette** (white → full spectrum → black)
4. **Nearest-color matching** (the math behind "best color" detection)
5. **Grayscale** conversion
6. **Black & White** (thresholding)
7. **Inverted** (negative) images
8. **Edge detection** — Sobel kernels from scratch
9. **The 2-D FFT** — seeing an image in the frequency domain
10. **Low-pass filtering** (blur) with FFT and Gaussian
11. **High-pass filtering** (edges) with FFT
12. Bonus: Sepia & how the Tkinter app ties everything together

**Requirements:** `pip install pillow numpy matplotlib`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageOps

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['image.cmap'] = 'gray'

## Step 1 — Load an image and look at it as numbers

A digital image is just a **3-D array of numbers**: `height × width × 3` (Red, Green, Blue), each value from **0 to 255** (`uint8`).

> 👉 Replace `'your_image.jpg'` with your own file. If you don't have one handy, run the cell below — it generates a colorful test image so the whole notebook works out of the box.

In [ ]:
# Option A: load your own image
# img = Image.open('your_image.jpg').convert('RGB')

# Option B: generate a synthetic test image (gradient + shapes)
w, h = 320, 240
x = np.linspace(0, 1, w)
y = np.linspace(0, 1, h)
X, Y = np.meshgrid(x, y)

arr = np.zeros((h, w, 3), dtype=np.uint8)
arr[..., 0] = (X * 255)                    # red increases left -> right
arr[..., 1] = (Y * 255)                    # green increases top -> bottom
arr[..., 2] = ((1 - X) * 255)              # blue decreases left -> right

# add a white circle and a black square (sharp edges for later steps!)
cy, cx, r = h // 2, w // 2, 40
mask = (np.arange(h)[:, None] - cy) ** 2 + (np.arange(w)[None, :] - cx) ** 2 <= r ** 2
arr[mask] = 255
arr[30:80, 30:80] = 0

img = Image.fromarray(arr)
print('Image size :', img.size)            # (width, height)
print('Array shape:', np.asarray(img).shape)  # (height, width, 3)
plt.imshow(img); plt.title('Our test image'); plt.axis('off'); plt.show()

## Step 2 — Pixels, RGB and HEX

Every pixel is a triplet `(R, G, B)`. The **HEX code** is just those three numbers written in base-16 (hexadecimal), two digits each:

$$\text{HEX} = \#\underbrace{RR}_{\text{red}}\underbrace{GG}_{\text{green}}\underbrace{BB}_{\text{blue}}$$

For example `(255, 165, 0)` → `#FFA500` (orange). This is exactly what the GUI shows when you move the mouse: it reads `array[y, x]` and formats it.

In [ ]:
arr = np.asarray(img)

# Pick a pixel — note: array indexing is [row, column] = [y, x]!
y_px, x_px = 120, 200
r, g, b = (int(v) for v in arr[y_px, x_px])

hex_code = f'#{r:02X}{g:02X}{b:02X}'
print(f'Pixel at (x={x_px}, y={y_px}):  RGB = ({r}, {g}, {b})  HEX = {hex_code}')

# Show the color as a swatch
plt.figure(figsize=(2, 2))
plt.imshow([[(r, g, b)]])
plt.title(hex_code); plt.axis('off'); plt.show()

## Step 3 — Build the 60-color reference palette

The app names colors by comparing each pixel to **60 reference colors** that span the whole spectrum **from white to black**:

- **48 chromatic colors** = 12 hues × 4 brightness/saturation variations
- **12 grayscale steps** = white → black

The easiest way to generate evenly-spaced hues is the **HSV color model** (Hue, Saturation, Value):
- **Hue** = position on the color wheel (0–360°) → we take 12 steps of 30°
- **Saturation** = how vivid the color is
- **Value** = how bright it is

In [ ]:
import colorsys

hue_names = ["Red", "Orange", "Yellow", "Chartreuse", "Green", "Spring Green",
             "Cyan", "Azure", "Blue", "Violet", "Magenta", "Rose"]

# (name prefix, saturation, value)
variations = [("Light ", 0.45, 1.00),
              ("",       1.00, 1.00),   # pure hue
              ("Dark ",  1.00, 0.65),
              ("Deep ",  1.00, 0.35)]

palette = []
for i, hname in enumerate(hue_names):
    hue = i / 12.0                       # 0.0 ... 11/12 around the wheel
    for prefix, s, v in variations:
        r, g, b = colorsys.hsv_to_rgb(hue, s, v)
        palette.append((prefix + hname, (round(r*255), round(g*255), round(b*255))))

gray_names = ["White", "Gray 90%", "Gray 80%", "Gray 70%", "Gray 60%", "Gray 50%",
              "Gray 40%", "Gray 30%", "Gray 20%", "Gray 10%", "Near Black", "Black"]
for name, lv in zip(gray_names, np.linspace(255, 0, 12).round().astype(int)):
    palette.append((name, (int(lv), int(lv), int(lv))))

print(f'Total colors: {len(palette)}')

# Visualize the palette as a 6x10 grid
fig, ax = plt.subplots(figsize=(12, 7))
for i, (name, rgb) in enumerate(palette):
    row, col = divmod(i, 10)
    ax.add_patch(plt.Rectangle((col, 5 - row), 0.95, 0.95,
                               color=np.array(rgb) / 255))
    ax.text(col + 0.47, 5 - row + 0.45, name, ha='center', va='center',
            fontsize=6.5, color='white' if sum(rgb) < 380 else 'black')
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')
ax.set_title('The 60-Color Reference Palette (white → spectrum → black)')
plt.show()

## Step 4 — Nearest-color matching ("best color")

When the cursor reads a pixel like `(203, 47, 61)`, which of the 60 names fits best?

We treat RGB as a point in 3-D space and find the palette color with the **smallest Euclidean distance**:

$$d = \sqrt{(R_1-R_2)^2 + (G_1-G_2)^2 + (B_1-B_2)^2}$$

The closest point wins. With NumPy we compute **all 60 distances at once** — no loop needed.

In [ ]:
palette_arr = np.array([rgb for _, rgb in palette], dtype=np.int32)   # shape (60, 3)

def nearest_color(rgb):
    diff = palette_arr - np.array(rgb, dtype=np.int32)   # (60, 3)
    dist2 = (diff ** 2).sum(axis=1)                      # squared distance, (60,)
    idx = int(np.argmin(dist2))
    name, prgb = palette[idx]
    return name, prgb, '#{:02X}{:02X}{:02X}'.format(*prgb)

# Try a few pixels
for test in [(203, 47, 61), (250, 250, 245), (10, 12, 8), (30, 100, 200), (240, 200, 90)]:
    name, prgb, hx = nearest_color(test)
    print(f'RGB {str(test):>17}  →  best match: {name:<14} {hx}  RGB{prgb}')

In [ ]:
# Visual check: input color vs. matched palette color
test_rgb = (203, 47, 61)
name, prgb, hx = nearest_color(test_rgb)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow([[test_rgb]]); axes[0].set_title(f'Input\nRGB{test_rgb}')
axes[1].imshow([[prgb]]);     axes[1].set_title(f'Best match: {name}\n{hx}')
for ax in axes: ax.axis('off')
plt.show()

## Step 5 — Grayscale

Human eyes are most sensitive to green, less to red, least to blue. So grayscale is a **weighted average** (ITU-R 601 standard, what PIL's `convert('L')` uses):

$$L = 0.299\,R + 0.587\,G + 0.114\,B$$

In [ ]:
arr = np.asarray(img, dtype=np.float64)

# Manual grayscale with the standard weights
gray_manual = arr @ np.array([0.299, 0.587, 0.114])

# PIL's built-in (same formula)
gray_pil = np.asarray(img.convert('L'))

fig, axes = plt.subplots(1, 3)
axes[0].imshow(img);          axes[0].set_title('Original')
axes[1].imshow(gray_manual);  axes[1].set_title('Manual: 0.299R+0.587G+0.114B')
axes[2].imshow(gray_pil);     axes[2].set_title("PIL convert('L')")
for ax in axes: ax.axis('off')
plt.show()

## Step 6 — Black & White (thresholding)

True black & white has only **two values**: 0 or 255. We pick a **threshold** `T` (often 128) and apply:

$$\text{output} = \begin{cases} 255 & \text{if pixel} \geq T \\ 0 & \text{otherwise} \end{cases}$$

Try changing `T` and watch how much of the image flips!

In [ ]:
gray = np.asarray(img.convert('L'))

fig, axes = plt.subplots(1, 3)
for ax, T in zip(axes, [80, 128, 180]):
    bw = np.where(gray >= T, 255, 0).astype(np.uint8)
    ax.imshow(bw); ax.set_title(f'Threshold T = {T}'); ax.axis('off')
plt.show()

## Step 7 — Inverted (negative)

Flipping every value is one subtraction:

$$\text{inverted} = 255 - \text{pixel}$$

White becomes black, red becomes cyan, etc.

In [ ]:
inverted = 255 - np.asarray(img)

fig, axes = plt.subplots(1, 2)
axes[0].imshow(img);      axes[0].set_title('Original')
axes[1].imshow(inverted); axes[1].set_title('Inverted: 255 − pixel')
for ax in axes: ax.axis('off')
plt.show()

## Step 8 — Edge detection with Sobel kernels (from scratch!)

An **edge** is a place where brightness changes quickly. We measure change with two small 3×3 **convolution kernels**:

$$K_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}
\qquad
K_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix}$$

- $K_x$ responds to **vertical** edges (horizontal change)
- $K_y$ responds to **horizontal** edges (vertical change)

**Convolution** = slide the kernel over every pixel, multiply-and-sum the 3×3 neighborhood. The edge strength is the gradient magnitude:

$$G = \sqrt{G_x^2 + G_y^2}$$

In [ ]:
gray = np.asarray(img.convert('L'), dtype=np.float64)

Kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)
Ky = Kx.T   # the y-kernel is just the transpose

# Pad the border so the output keeps the same size
padded = np.pad(gray, 1, mode='edge')

# sliding_window_view gives every 3x3 neighborhood -> shape (H, W, 3, 3)
windows = np.lib.stride_tricks.sliding_window_view(padded, (3, 3))

Gx = np.einsum('ijkl,kl->ij', windows, Kx)   # convolve with Kx
Gy = np.einsum('ijkl,kl->ij', windows, Ky)   # convolve with Ky
G  = np.hypot(Gx, Gy)                        # sqrt(Gx² + Gy²)
G  = G / G.max() * 255

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(gray);        axes[0].set_title('Grayscale input')
axes[1].imshow(np.abs(Gx));  axes[1].set_title('|Gx| — vertical edges')
axes[2].imshow(np.abs(Gy));  axes[2].set_title('|Gy| — horizontal edges')
axes[3].imshow(G);           axes[3].set_title('Magnitude √(Gx²+Gy²)')
for ax in axes: ax.axis('off')
plt.show()

## Step 9 — The 2-D FFT: an image in the frequency domain

The **Fast Fourier Transform** decomposes an image into **sine waves of different frequencies**:

- **Low frequencies** (near the center after `fftshift`) = smooth regions, overall shapes
- **High frequencies** (far from center) = edges, fine detail, noise

Workflow:
1. `np.fft.fft2(gray)` → complex frequency coefficients
2. `np.fft.fftshift(...)` → move the zero frequency to the **center**
3. Plot `log(1 + |F|)` — the log makes the huge dynamic range visible

In [ ]:
gray = np.asarray(img.convert('L'), dtype=np.float64)

F = np.fft.fft2(gray)            # 2-D FFT (complex numbers!)
F_shifted = np.fft.fftshift(F)   # zero-frequency -> center
magnitude = np.log1p(np.abs(F_shifted))   # log(1 + |F|)

fig, axes = plt.subplots(1, 2)
axes[0].imshow(gray);      axes[0].set_title('Spatial domain (the image)')
axes[1].imshow(magnitude); axes[1].set_title('Frequency domain (log magnitude)')
for ax in axes: ax.axis('off')
plt.show()

print('Center = low frequencies (smooth areas).')
print('Bright lines/spots away from center = strong edges & repeating patterns.')

## Step 10 — Low-pass filter (FFT blur)

A **low-pass filter** keeps low frequencies and removes high ones → the image gets **blurry** (detail and noise live in high frequencies).

Recipe:
1. FFT + shift
2. Multiply by a **circular mask** — 1 inside a radius around the center, 0 outside
3. Inverse shift + inverse FFT → back to a (blurred) image

Smaller radius ⇒ fewer frequencies kept ⇒ blurrier.

In [ ]:
def fft_low_pass(gray, radius_ratio):
    F = np.fft.fftshift(np.fft.fft2(gray))
    h, w = gray.shape
    cy, cx = h // 2, w // 2
    Y, X = np.ogrid[:h, :w]
    radius = radius_ratio * min(h, w)
    mask = (Y - cy)**2 + (X - cx)**2 <= radius**2     # circle of 1s
    out = np.fft.ifft2(np.fft.ifftshift(F * mask)).real
    return np.clip(out, 0, 255), mask

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for col, ratio in enumerate([0.30, 0.10, 0.04]):
    filtered, mask = fft_low_pass(gray, ratio)
    axes[0, col].imshow(mask);     axes[0, col].set_title(f'Mask, radius={ratio:.0%} of size')
    axes[1, col].imshow(filtered); axes[1, col].set_title('Result (inverse FFT)')
for ax in axes.flat: ax.axis('off')
plt.suptitle('Low-pass filtering: smaller circle → blurrier image', y=0.98)
plt.show()

### Gaussian blur — the spatial-domain cousin

A Gaussian blur does (almost) the same job by **averaging each pixel with its neighbors**, weighted by a bell curve. It's a low-pass filter too — just computed in the spatial domain instead of the frequency domain.

In [ ]:
fig, axes = plt.subplots(1, 3)
axes[0].imshow(img); axes[0].set_title('Original')
for ax, r in zip(axes[1:], [2, 6]):
    ax.imshow(img.filter(ImageFilter.GaussianBlur(r)))
    ax.set_title(f'GaussianBlur(radius={r})')
for ax in axes: ax.axis('off')
plt.show()

## Step 11 — High-pass filter (the opposite)

Invert the mask — **block** the low-frequency center, **keep** the rest. What survives is exactly what the low-pass removed: **edges and fine detail**. This is why high-pass output looks like an edge map.

In [ ]:
def fft_high_pass(gray, radius_ratio=0.05):
    F = np.fft.fftshift(np.fft.fft2(gray))
    h, w = gray.shape
    cy, cx = h // 2, w // 2
    Y, X = np.ogrid[:h, :w]
    radius = radius_ratio * min(h, w)
    mask = (Y - cy)**2 + (X - cx)**2 > radius**2      # everything EXCEPT the center
    out = np.abs(np.fft.ifft2(np.fft.ifftshift(F * mask)).real)
    return out / out.max() * 255

fig, axes = plt.subplots(1, 3)
axes[0].imshow(gray);                      axes[0].set_title('Original (gray)')
axes[1].imshow(fft_high_pass(gray, 0.03)); axes[1].set_title('High-pass (small block)')
axes[2].imshow(fft_high_pass(gray, 0.10)); axes[2].set_title('High-pass (bigger block)')
for ax in axes: ax.axis('off')
plt.show()

## Step 12 — Bonus: Sepia (a color matrix)

Many color effects are a single **matrix multiplication** applied to every RGB pixel. Sepia uses this classic matrix:

$$\begin{bmatrix} R' \\ G' \\ B' \end{bmatrix} =
\begin{bmatrix} 0.393 & 0.769 & 0.189 \\ 0.349 & 0.686 & 0.168 \\ 0.272 & 0.534 & 0.131 \end{bmatrix}
\begin{bmatrix} R \\ G \\ B \end{bmatrix}$$

In [ ]:
M = np.array([[0.393, 0.769, 0.189],
              [0.349, 0.686, 0.168],
              [0.272, 0.534, 0.131]])

arr = np.asarray(img, dtype=np.float64)
sepia = np.clip(arr @ M.T, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2)
axes[0].imshow(img);   axes[0].set_title('Original')
axes[1].imshow(sepia); axes[1].set_title('Sepia (matrix transform)')
for ax in axes: ax.axis('off')
plt.show()

## Step 13 — How the Tkinter app ties it all together

Open `color_detector_app.py` alongside this notebook. The key connections:

| GUI feature | Technique from this notebook | Where in the app |
|---|---|---|
| 📂 Load Image button | `Image.open(...)` via `filedialog` | `load_image()` |
| Cursor color readout | `array[y, x]` → HEX formatting (Step 2) | `on_mouse_move()` |
| "Best match of 60" | palette (Step 3) + nearest color (Step 4) | `nearest_color()` |
| Grayscale / B&W / Invert | Steps 5–7 | `FILTERS` dict |
| Edge Detection (Sobel) | Step 8 — same `einsum` convolution | `sobel_edges()` |
| FFT Spectrum | Step 9 | `fft_spectrum()` |
| Low Pass (FFT / Gaussian) | Step 10 | `fft_low_pass()` |
| High Pass (FFT) | Step 11 | `fft_high_pass()` |
| Sepia | Step 12 | `sepia()` |

The Tkinter pattern is simple:
```python
combo.bind("<<ComboboxSelected>>", lambda e: self.apply_filter())  # dropdown -> filter
self.canvas.bind("<Motion>", self.on_mouse_move)                   # mouse -> color
```
Each filter is just a function `PIL.Image -> PIL.Image` stored in a dictionary, so the dropdown only needs one line: `FILTERS[name](image)`.

### 🧪 Exercises
1. Add a **Sobel threshold** slider: show only edges where `G > T`.
2. Change the low-pass mask from a hard circle to a **Gaussian** mask — the ringing artifacts disappear. Why?
3. Extend the palette to **120 colors** (6 variations per hue). Does nearest-matching get noticeably better?
4. Replace Euclidean RGB distance with a **weighted** distance (e.g., weights 2, 4, 3 for R, G, B). Does it match human perception better?
